# **Uploading files & combining data tables**

In [ ]:
# Importing tools
import pandas as pd
import glob
import numpy as np
from google.colab import files

# Upload Crime Data
# Source: https://geodash.vpd.ca/opendata/
uploaded = files.upload()

Saving crimedata_csv_AllNeighbourhoods_2026.csv to crimedata_csv_AllNeighbourhoods_2026 (1).csv
Saving crimedata_csv_AllNeighbourhoods_2025.csv to crimedata_csv_AllNeighbourhoods_2025.csv


In [ ]:
# Checking files
csv_files = list(uploaded.keys())
print(csv_files)

['crimedata_csv_AllNeighbourhoods_2026 (1).csv', 'crimedata_csv_AllNeighbourhoods_2025.csv']


In [ ]:
Van_df_2025 = pd.read_csv(csv_files[0])
Van_df_2026 = pd.read_csv(csv_files[1])
Van_df = pd.merge(Van_df_2025, Van_df_2026, how='outer')
print(Van_df.shape)
Van_df.head()

(52956, 10)


,TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y
0,Break and Enter Commercial,2025,1,1,0,0,26XX RUPERT ST,Renfrew-Collingwood,497554.6183,5.456431e+06
1,Break and Enter Commercial,2025,1,1,0,1,14XX W BROADWAY AVE,Fairview,490043.3810,5.456761e+06
2,Break and Enter Commercial,2025,1,1,18,50,7XX GREAT NORTHERN WAY,Mount Pleasant,493622.5528,5.457073e+06
3,Break and Enter Commercial,2025,1,1,19,28,3XX E PENDER ST,Strathcona,492991.7908,5.458618e+06
4,Break and Enter Commercial,2025,1,2,0,0,10XX PACIFIC ST,West End,490258.8902,5.458368e+06


# Removing Duplicates (Rows with NaN Crime Data are not included)

In [ ]:
# Check total number of rows
print('Total Rows:', Van_df.shape)

Total Rows: (52956, 10)


In [ ]:
Van_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52956 entries, 0 to 52955
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TYPE           52956 non-null  object 
 1   YEAR           52956 non-null  int64  
 2   MONTH          52956 non-null  int64  
 3   DAY            52956 non-null  int64  
 4   HOUR           52956 non-null  int64  
 5   MINUTE         52956 non-null  int64  
 6   HUNDRED_BLOCK  52956 non-null  object 
 7   NEIGHBOURHOOD  52946 non-null  object 
 8   X              52956 non-null  float64
 9   Y              52956 non-null  float64
dtypes: float64(2), int64(5), object(3)
memory usage: 4.0+ MB


In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:32610", "EPSG:4326", always_xy=True)

lats = []
longs = []

for index, row in Van_df.iterrows():
    x = row['X']
    y = row['Y']
    lon, lat = transformer.transform(x, y)
    lats.append(round(lat,5))
    longs.append(round(lon,5))

Van_df['Lat'] = lats
Van_df['Long'] = longs

print(Van_df[['X', 'Y', 'Lat', 'Long']].head())

             X             Y       Lat       Long
0  497554.6183  5.456431e+06  49.26064 -123.03361
1  490043.3810  5.456761e+06  49.26353 -123.13685
2  493622.5528  5.457073e+06  49.26638 -123.08766
3  492991.7908  5.458618e+06  49.28028 -123.09636
4  490258.8902  5.458368e+06  49.27799 -123.13393


In [ ]:
# Check how many rows are duplicates
print(Van_df.duplicated().sum())

2039


In [ ]:
# Removing duplicates from the has_id table
Van_df = Van_df.drop_duplicates(keep='last')
print('After deduplicate:', Van_df.shape)

After deduplicate: (50917, 12)


In [ ]:
# Checking the table
Van_df.head()

,TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y,Lat,Long
0,Break and Enter Commercial,2025,1,1,0,0,26XX RUPERT ST,Renfrew-Collingwood,497554.6183,5.456431e+06,49.26064,-123.03361
1,Break and Enter Commercial,2025,1,1,0,1,14XX W BROADWAY AVE,Fairview,490043.3810,5.456761e+06,49.26353,-123.13685
2,Break and Enter Commercial,2025,1,1,18,50,7XX GREAT NORTHERN WAY,Mount Pleasant,493622.5528,5.457073e+06,49.26638,-123.08766
3,Break and Enter Commercial,2025,1,1,19,28,3XX E PENDER ST,Strathcona,492991.7908,5.458618e+06,49.28028,-123.09636
4,Break and Enter Commercial,2025,1,2,0,0,10XX PACIFIC ST,West End,490258.8902,5.458368e+06,49.27799,-123.13393


# Filtering data based on a 0.5 KM radius from Vancouver Office (Using haversine)

In [ ]:
def haversine_dis(lat1, long1, lat2, long2):
  # Covert degrees to radians
  lat1, long1, lat2, long2 = map(np.radians, [lat1, long1, lat2, long2])

  # Differences
  dlat = lat2 - lat1
  dlong = long2 - long1

  # Haversine formuala
  a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlong / 2)**2
  c = 2 * np.arcsin(np.sqrt(a))

  r = 6371 # Radius of Earth in KM
  return r * c

# Lat/Long of Vancouver Office
Lon_lat = 49.285
Lon_long = -123.119

# Calulating the distance of every crime data point to the Vancouver Office. The calculated KM stored into a new column
Van_df['distance_km'] = haversine_dis(
      Lon_lat, Lon_long, Van_df['Lat'], Van_df['Long']
).round(2)

print(Van_df.shape)
Van_df.head()

(50917, 13)


,TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y,Lat,Long,distance_km
0,Break and Enter Commercial,2025,1,1,0,0,26XX RUPERT ST,Renfrew-Collingwood,497554.6183,5.456431e+06,49.26064,-123.03361,6.76
1,Break and Enter Commercial,2025,1,1,0,1,14XX W BROADWAY AVE,Fairview,490043.3810,5.456761e+06,49.26353,-123.13685,2.72
2,Break and Enter Commercial,2025,1,1,18,50,7XX GREAT NORTHERN WAY,Mount Pleasant,493622.5528,5.457073e+06,49.26638,-123.08766,3.08
3,Break and Enter Commercial,2025,1,1,19,28,3XX E PENDER ST,Strathcona,492991.7908,5.458618e+06,49.28028,-123.09636,1.72
4,Break and Enter Commercial,2025,1,2,0,0,10XX PACIFIC ST,West End,490258.8902,5.458368e+06,49.27799,-123.13393,1.33


In [ ]:
# Filter all crime outside of a 0.5 KM radius
Van_df = Van_df[Van_df['distance_km'] <= 0.5]

# Check table again
print(Van_df.shape)
Van_df.head()

(6894, 13)


,TYPE,YEAR,MONTH,DAY,HOUR,MINUTE,HUNDRED_BLOCK,NEIGHBOURHOOD,X,Y,Lat,Long,distance_km
45,Break and Enter Commercial,2025,1,13,3,43,10XX ROBSON ST,West End,491036.3006,5.458994e+06,49.28363,-123.12325,0.34
81,Break and Enter Commercial,2025,1,23,14,34,7XX THURLOW ST,West End,490968.9291,5.459157e+06,49.28510,-123.12418,0.38
116,Break and Enter Commercial,2025,2,4,3,8,11XX ROBSON ST,West End,490909.5334,5.459106e+06,49.28464,-123.12500,0.44
125,Break and Enter Commercial,2025,2,7,4,30,7XX THURLOW ST,West End,490943.1339,5.459118e+06,49.28475,-123.12454,0.40
126,Break and Enter Commercial,2025,2,7,11,10,7XX THURLOW ST,West End,490943.1339,5.459118e+06,49.28475,-123.12454,0.40


# Export table as a CSV

In [ ]:
out_path = "Vancouver_Office_Crime_Data_2025_-_2026.csv"
Van_df.to_csv(out_path, index=False)